## Apresentação 

Notebook destinado ao estudo da utilização de modelos clássicos de ML para realizar a classificação de mensagens segundo seus temas correlacionados. No presente caso, o conjunto de dados conterá perguntas relativas a cada anime - as quais serão as features, enquanto os animes as labels. O objetivo, portanto, é compreender se os modelos apreciados conseguem predizer o tema do anime para a label informada. A importância desse tipo de predição se relaciona a diversos contextos dentro da ciência de dados, como análise de sentimento, mas também podem servir de utilização para o uso de um filtro para o escopo do Retrieval Argumentend Generation (RAG), permitindo a definição de um subconjunto na busca, diminuindo a sua incerteza latente - ver sobre teoria da informação, Claude Shannon. 

##### Metodologia

Como metodologia foi utilizado o modelo GPT-4 para a síntese das mensagens. A escolha de uma quantidade baixa de mensagens se dá em direção em conceber se a menor quantidade de dados, presente em alguns contextos, podem já ser últi ou não a tarefa de classificação. Os modelos escolhidos para classificar as mensagens serão : XGBoost, Random Forest e SVM. A avaliação dos modelos irá ocorrer pelo uso de precision, recall e f1-score, bem como da curva ROC-AUC. Iicialmente existirá uma primeira `pipe` ou run desses modelos nos dados de treino, que servirão como baseline, para depois serem avaliados segundo os seus respectivos análogos otimizados aos dados. A otimização ocorrerá via o método RandomSearch. 

### Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import spacy
import umap
import xgboost as xgb

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm

In [3]:
SEED = 15

In [4]:
nlp = spacy.load("pt_core_news_sm")

In [5]:
# Testando a remoção das stopwords :

text = "A andorinha, Ciri, era o amor do lobo branco, a qual era vista como a sua adorável filha"

new_text = " ".join([token.text for token in nlp(text=text) if not token.is_stop and not token.is_punct])
new_text

'andorinha Ciri amor lobo branco vista adorável filha'

### Carregando o dataset

In [6]:
df = pd.read_csv("anime_questions_dataset.csv")
df.head()

,perguntas,label
0,O que é um ghoul exatamente e como eles se ali...,Tokyo Ghoul
1,Quem é Ken Kaneki e qual é a importância da má...,Tokyo Ghoul
2,Como funciona a organização Anteiku? Vale a pe...,Tokyo Ghoul
3,Quais são as diferenças principais entre o ani...,Tokyo Ghoul
4,O que significa “kakuhou” e por que é importan...,Tokyo Ghoul


In [7]:
df.tail()

,perguntas,label
115,Pode indicar filmes que tratem bullying e rede...,Koe no Katachi
116,Como explicar para alguém que acha o protagoni...,Koe no Katachi
117,Há cenas que fãs consideram as “mais bonitas” ...,Koe no Katachi
118,Como o silêncio (ausência de fala) é usado com...,Koe no Katachi
119,"Se eu quisesse discutir o filme em grupo, que ...",Koe no Katachi


In [8]:
df.isnull().sum()

perguntas    0
label        0
dtype: int64

In [9]:
df.shape

(120, 2)

### Dataprep

Como preparação dos dados para a utilização dos algoritmos de ML para fins de classficação, esses serão tratados e normalizados. Nesse sentido será utilizado a remoção das `stopwords` (palavras que podem ser entendidas como artigos e preposições, por exemplo, que estão presentes nos textos), bem como a redução dos termos segundo os seus 'lemas', isto é, ao seu radical que as caracterizam como tal, o significante para um determinado significado. Ex: o radical da palavra andando seria andar, de tal modo que o primeiro termo reduzido ao seu lema, no texto, passaria a se apresentar apenas como andar. 

Após esse processo, as mensagens irão passar por um processo de vetorização, via TF-IDF e por uso de modelos de embeddings. Para os vetores formados via modelo de embedding, existirão três verões : uma sem redução de dimensionalidade, outra com redução via PCA e via UMAP. A principal diferença do PCA em relação ao UMAP é que o primeiro assume uma relação linear entre as dimensões para a identificação dos componentes principais, enquanto o outro não, que visa conservar o formato local e global dos vetores durante a redução.

**Obs :** Para fins de estudo, será três versões de tais textos concebidos segundo ao seus níveis de tratamento. As versões terá uma sem tratamento, outra com apenas remoção das stopwords e a terceira com lematização.  

In [10]:
def remove_stop_words(sentence: str) -> str: 
    """
    Função que recebe um texto e remove 
    as suas stopwords presentes. 
    """
    text = sentence.lower()
    return " ".join([token.text for token in nlp(text=text) if not token.is_stop and not token.is_punct])
    

In [11]:
def lemma_apply(sentence: str) -> str:
    """
    Função que reduz a palavra ao seu respectivo lema
    """
    text = nlp(sentence)
    return " ".join([token.lemma_ for token in text if not token.is_stop and not token.is_punct])

In [12]:
def clean_text(sentence: str) -> str:
    """    
    Função que limpa o texto, removendo tanto stopwords quanto 
    reduz às palavras aos seus respectivos lemas. 
    """
    text_without_stopwords = remove_stop_words(sentence=sentence)
    text_with_lemas = lemma_apply(sentence=text_without_stopwords)
    return text_with_lemas

- Feature sem stopwords

In [13]:
text_without_stopwords = []

for i in tqdm(range(df.shape[0]), desc="Removendo stopwords"):

    clean_text = remove_stop_words(df["perguntas"][i])
    text_without_stopwords.append(clean_text)

Removendo stopwords: 100%|██████████| 120/120 [00:00<00:00, 121.01it/s]


In [14]:
df["text_without_stopwords"] = text_without_stopwords

In [15]:
df["perguntas"][1]

'Quem é Ken Kaneki e qual é a importância da máscara dele?'

In [16]:
df["text_without_stopwords"][3]

'diferenças principais anime mangá tokyo ghoul'

- Feature com lemas

In [17]:
text_wit_lematization = []

for i in tqdm(range(df.shape[0]), desc="Aplicando lematização"):

    clean_text = lemma_apply(df["perguntas"][i])
    text_wit_lematization.append(clean_text)

Aplicando lematização: 100%|██████████| 120/120 [00:00<00:00, 127.25it/s]


In [18]:
df["text_with_lemas"] = text_wit_lematization

In [19]:
df["perguntas"][0]

'O que é um ghoul exatamente e como eles se alimentam? (sem spoilers)'

In [20]:
df["text_with_lemas"][3]

'diferença principal anime mangá Tokyo Ghoul'

- Sem stopwords e com lematization

In [ ]:
clean_text_list = []

for i in tqdm(range(df.shape[0]), desc="Limpando o texto"):

    clean_text_ = clean_text(df["perguntas"][i])
    clean_text_list.append(clean_text_)

In [ ]:
df["clean_text"] = clean_text_list

In [24]:
df.head()

,perguntas,label,text_without_stopwords,text_with_lemas
0,O que é um ghoul exatamente e como eles se ali...,Tokyo Ghoul,ghoul exatamente alimentam spoilers,ghoul exatamente alimentar spoiler
1,Quem é Ken Kaneki e qual é a importância da má...,Tokyo Ghoul,ken kaneki importância máscara dele,Ken Kaneki importância máscara de ele
2,Como funciona a organização Anteiku? Vale a pe...,Tokyo Ghoul,funciona organização anteiku vale pena ler mangá,funcionar organização Anteiku Vale pena ler mangá
3,Quais são as diferenças principais entre o ani...,Tokyo Ghoul,diferenças principais anime mangá tokyo ghoul,diferença principal anime mangá Tokyo Ghoul
4,O que significa “kakuhou” e por que é importan...,Tokyo Ghoul,significa kakuhou importante lutas,significar kakuhar importante luta


- Formatando as labels

In [25]:
encoder = LabelEncoder()

label_encoded = encoder.fit_transform(df["label"])

In [26]:
df["label_encoded"] = label_encoded

#### Vetorizando as mensagens

- TF-IDF

In [ ]:
tf_idf = TfidfVectorizer()

tf_idf_vector_messages = tf_idf.fit_transform(df["perguntas"])
tf_idf_vector_messages_2 = tf_idf.fit_transform(df["text_with_lemas"])
tf_idf_vector_messages_3 = tf_idf.fit_transform(df["text_without_stopwords"])

- Embeddings

In [27]:
%%time

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

CPU times: total: 9.41 s
Wall time: 43 s


In [28]:
def make_embeddings(sentence: str) -> list:
    """
    Função que cria vetores dos textos informados
    """
    return embeddings.embed_query(sentence)

In [29]:
embeddings_raw_text = []

for i in tqdm(range(df.shape[0]), desc="Formando embeddings"):

    embedding_text = make_embeddings(df["perguntas"][i])
    embeddings_raw_text.append(embedding_text)

Formando embeddings: 100%|██████████| 120/120 [00:14<00:00,  8.33it/s]


In [30]:
embeddings_no_stopwords_text = []

for i in tqdm(range(df.shape[0]), desc="Formando embeddings"):

    embedding_text = make_embeddings(df["text_without_stopwords"][i])
    embeddings_no_stopwords_text.append(embedding_text)

Formando embeddings: 100%|██████████| 120/120 [00:08<00:00, 13.60it/s]


In [31]:
embeddings_lematization_text = []

for i in tqdm(range(df.shape[0]), desc="Formando embeddings"):

    embedding_text = make_embeddings(df["text_with_lemas"][i])
    embeddings_lematization_text.append(embedding_text)

Formando embeddings: 100%|██████████| 120/120 [00:08<00:00, 13.75it/s]


In [32]:
df_vector = pd.DataFrame(
    {
        "Embedding Raw Text": embeddings_raw_text, 
        "Embedding no Stopword Text": embeddings_no_stopwords_text, 
        "Embedding Lematization Text": embeddings_lematization_text
    }
)

In [33]:
df_vector["label_encoded"] = label_encoded

In [34]:
df_vector.head(2)

,Embedding Raw Text,Embedding no Stopword Text,Embedding Lematization Text,label_encoded
0,"[0.02300974726676941, -0.026236355304718018, 0...","[1.2414464436005801e-05, -0.02159101516008377,...","[-0.010516703128814697, -0.01708705723285675, ...",4
1,"[0.010023861192166805, -0.018375582993030548, ...","[-0.00031401313026435673, 0.027115844190120697...","[0.0042190225794911385, 0.010120215825736523, ...",4


In [35]:
df_vector.shape

(120, 4)

- Scaler

In [ ]:
def scaler(vectors: list) -> list: 
    """ 
    Função que normaliza os vetores, para que esses sejam passados
    depois para os algortimos de classificação. 
    """

    # Transformando as listas dos embeddings em matrizes de duas 
    # dimensões, para que possa ser enviado ao algoritmo. 
    array_vectors = np.vstack(vectors)
    
    scaler = StandardScaler()
    sacaled_vectors = scaler.fit_transform(array_vectors)
    return sacaled_vectors

- PCA

In [99]:
def plot_pca_elbow_plotly(vectors, max_components=0):
    """
    Plota o gráfico de variância explicada cumulativa usando Plotly.
    
    Args:
        X: array-like, shape (n_samples, n_features)
        max_components: número máximo de componentes a avaliar
    """

    X_scaled = scaler(vectors=vectors)
    
    pca = PCA(n_components=max_components)
    pca.fit(X_scaled)
    
    explained_variance_ratio = np.cumsum(pca.explained_variance_ratio_)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(range(1, max_components+1)),
        y=explained_variance_ratio,
        mode='lines+markers',
        name='Variância Cumulativa Explicada'
    ))
    
    fig.update_layout(
        title="Elbow Method para PCA",
        xaxis_title="Número de Componentes",
        yaxis_title="Variância Cumulativa Explicada",
        xaxis=dict(tickmode='linear'),
        yaxis=dict(range=[0,1]),
        template='plotly_white'
    )
    
    fig.show()

In [97]:
plot_pca_elbow_plotly(vectors=df_vector["Embedding Raw Text"])

- UMAP

In [148]:
import umap

In [ ]:
umap_reducer = umap.UMAP()

### Dividindo a porção em treino e teste

In [ ]:
y = df_vector["label_encoded"]

X_raw_text          = df_vector["Embedding Raw Text"]
X_lema_text         = df_vector["Embedding Lematization Text"]
X_no_stopwords_text = df_vector["Embedding no Stopword Text"]

In [46]:
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_raw_text, y, test_size=0.20, random_state=SEED, stratify=y)
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_lema_text, y, test_size=0.20, random_state=SEED, stratify=y)
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_no_stopwords_text, y, test_size=0.20, random_state=SEED, stratify=y)

#### Normalizando os conjuntos de treino e teste

A normalização dos vetores das mensagens corresponde a uma forma de reduzir a influência de diferentes escalas desses para os algortimos de classificação, sujeitando todos os valores numa mesma escala. A realização da normalização após a divisão ocorre em razão de buscar impedir um fenômeno de data leakage. 

In [153]:
scaled_X_train_1 = scaler(vectors=X_train_1)
scaled_X_train_2 = scaler(vectors=X_train_2)
scaled_X_train_3 = scaler(vectors=X_train_3)

scaled_X_test_1 = scaler(vectors=X_test_1)
scaled_X_test_2 = scaler(vectors=X_test_2)
scaled_X_test_3 = scaler(vectors=X_test_3)

In [116]:
pca = PCA(n_components=50)

X_train_1_pca     = pca.fit_transform(scaler(vectors=X_train_1))
X_train_2_pca    = pca.fit_transform(scaler(vectors=X_train_2))
X_train_3_pca = pca.fit_transform(scaler(vectors=X_train_3))

X_test_1_pca     = pca.transform(scaler(X_test_1))
X_test_2_pca    = pca.transform(scaler(X_test_2))
X_test_3_pca = pca.transform(scaler(X_test_3))

### Modelos

Instanciando os modelos considerados e encontrando métricas de baseline para eles - nesse sentido, eles estão sendo em sua forma padrão, sem demais ajustes de parametrização que visam a sua otimização aos dados.

In [66]:
def metrics(predict, ground_truth) -> dict:
    """  
    Função que realiza o cálculo das métricas do modelo ajustado.
    """
    precision = precision_score(ground_truth, predict, average="weighted")
    recall = recall_score(ground_truth, predict, average="weighted")
    f1 = f1_score(ground_truth, predict, average="weighted")

    return {
        "precision": round(precision, 3), 
        "recall": round(recall, 3), 
        "f1": round(f1, 3)
    }

- XGBoost - Baseline

In [ ]:
%%time

xgboost_classifier = xgb.XGBClassifier()

xgboost_classifier.fit(new_X_train_1, y_train_1)
y_xgboost_1 = xgboost_classifier.predict(new_X_test_1)

xgboost_classifier.fit(new_X_train_2, y_train_2)
y_xgboost_2 = xgboost_classifier.predict(new_X_test_2)

xgboost_classifier.fit(new_X_train_3, y_train_3)
y_xgboost_3 = xgboost_classifier.predict(new_X_test_3)

CPU times: total: 14.5 s
Wall time: 2.81 s


In [ ]:
%%time

# Com PCA

xgboost_classifier = xgb.XGBClassifier()

xgboost_classifier.fit(X_train_1_pca, y_train_1)
y_xgboost_1_pca = xgboost_classifier.predict(X_test_1_pca)

xgboost_classifier.fit(X_train_2_pca, y_train_2)
y_xgboost_2_pca = xgboost_classifier.predict(X_test_2_pca)

xgboost_classifier.fit(X_train_3_pca, y_train_3)
y_xgboost_3_pca = xgboost_classifier.predict(X_test_3_pca)

CPU times: total: 3.09 s
Wall time: 731 ms


In [132]:
print(f"""\
Sem PCA:
{
    metrics(
        predict      = y_xgboost_1, 
        ground_truth = y_test_1
    )
}
"""
)

Sem PCA:
{'precision': 0.357, 'recall': 0.333, 'f1': 0.34}



In [134]:
print(f"""\
Com PCA:
{
    metrics(
        predict      = y_xgboost_3_pca, 
        ground_truth = y_test_3
    )
}
"""
)

Com PCA:
{'precision': 0.325, 'recall': 0.25, 'f1': 0.261}



- Random Forest - Baseline

In [70]:
%%time

random_forest = RandomForestClassifier(random_state=SEED)

random_forest.fit(new_X_train_1, y_train_1)
y_random_forest_1 = random_forest.predict(new_X_test_1)

random_forest.fit(new_X_train_2, y_train_2)
y_random_forest_2 = random_forest.predict(new_X_test_2)

random_forest.fit(new_X_train_3, y_train_3)
y_random_forest_3 = random_forest.predict(new_X_test_3)

CPU times: total: 641 ms
Wall time: 616 ms


In [ ]:
%%time

# Com PCA

random_forest = RandomForestClassifier(random_state=SEED)

random_forest.fit(X_train_1_pca, y_train_1)
y_random_forest_1_pca = random_forest.predict(X_test_1_pca)

random_forest.fit(X_train_2_pca, y_train_2)
y_random_forest_2_pca = random_forest.predict(X_test_2_pca)

random_forest.fit(X_train_3_pca, y_train_3)
y_random_forest_3_pca = random_forest.predict(X_test_3_pca)

CPU times: total: 453 ms
Wall time: 602 ms


In [136]:
print(f"""\
Sem PCA:
{
    metrics(
        predict      = y_random_forest_1, 
        ground_truth = y_test_1
    )
}
"""
)

Sem PCA:
{'precision': 0.472, 'recall': 0.417, 'f1': 0.431}



In [142]:
print(f"""\
Com PCA:
{
    metrics(
        predict      = y_random_forest_1_pca, 
        ground_truth = y_test_1
    )
}
"""
)

Com PCA:
{'precision': 0.021, 'recall': 0.042, 'f1': 0.028}



- RandomForest - Tuning

In [204]:
%%time 

param_dist_rf = {
    "n_estimators": [50, 100, 150, 200, 300],
    "max_depth": [2, 5, 10, 15],
    "criterion": ["gini", "entropy", "log_loss"],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [2, 4, 6, 8, 10],
}

rf_search = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_dist_rf,
    n_iter=200,  # número de combinações aleatórias testadas
    cv=5,       # cross-validation 5-fold
    scoring='f1_weighted',  # métrica
    verbose=1,
    n_jobs=-1,  # usa todos os núcleos
    random_state=SEED
)

rf_search.fit(scaled_X_train_1, y_train_1)

Fitting 5 folds for each of 200 candidates, totalling 1000 fits
CPU times: total: 2.94 s
Wall time: 1min 6s


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(max_depth=5,
                                                    min_samples_split=5,
                                                    n_estimators=300,
                                                    random_state=15),
                   n_iter=200, n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy',
                                                      'log_loss'],
                                        'max_depth': [2, 5, 10, 15],
                                        'min_samples_leaf': [2, 4, 6, 8, 10],
                                        'min_samples_split': [2, 5, 10, 15],
                                        'n_estimators': [50, 100, 150, 200,
                                                         300]},
                   random_state=15, scoring='f1_weighted', verbose=1)

In [206]:
print("Melhores hiperparâmetros Random Forest:", rf_search.best_params_)
print("Melhor F1-weighted:", rf_search.best_score_)

Melhores hiperparâmetros Random Forest: {'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_depth': 2, 'criterion': 'gini'}
Melhor F1-weighted: 0.2993684210526316


In [247]:
random_forest_tuning = RandomForestClassifier(
    n_estimators=150,
    criterion="gini", 
    min_samples_split=10, 
    min_samples_leaf=5, 
    max_depth=10,
    random_state=SEED
)

In [254]:
random_forest_tuning.fit(scaled_X_train_1, y_train_1)
y_random_forest_tuning_1 = random_forest_tuning.predict(scaled_X_test_1)

random_forest_tuning.fit(scaled_X_train_2, y_train_2)
y_random_forest_tuning_2 = random_forest_tuning.predict(scaled_X_test_2)

random_forest_tuning.fit(scaled_X_train_3, y_train_3)
y_random_forest_tuning_3 = random_forest_tuning.predict(scaled_X_test_3)

In [255]:
print(f"""\
Sem PCA e com otimização:
{
    metrics(
        predict      = y_random_forest_tuning_3, 
        ground_truth = y_test_3
    )
}
"""
)

Sem PCA e com otimização:
{'precision': 0.427, 'recall': 0.375, 'f1': 0.353}



- SVM - Baseline

In [75]:
%%time

smv = SVC(random_state=SEED)

smv.fit(new_X_train_1, y_train_1)
y_smv_1 = smv.predict(new_X_test_1)

smv.fit(new_X_train_2, y_train_2)
y_smv_2 = smv.predict(new_X_test_2)

smv.fit(new_X_train_3, y_train_3)
y_smv_3 = smv.predict(new_X_test_3)

CPU times: total: 31.2 ms
Wall time: 155 ms


In [79]:
metrics(
    predict      = y_smv_1, 
    ground_truth = y_test_1
)

{'precision': 0.347, 'recall': 0.292, 'f1': 0.303}